# The Discrete Logarithm and Diffie–Hellman Key Exchange

This notebook meets a logarithm that is **hard** to compute. In modular arithmetic, raising a number to a power is fast, but recovering the exponent — the **discrete logarithm** — seems to require enormous effort. That one-way asymmetry lets two people agree on a secret key while an eavesdropper hears every word. We implement everything at toy sizes, measure how the work grows, and see why the real logarithm is easy while the discrete one is not.

This is notebook 05 of module 06 (**Applications**) in the **Learning Logarithms** series. All numbers here are deliberately tiny and for learning only — never use this code to protect anything.

## 1. Introduction

Alice and Bob want to talk privately over the internet. To encrypt, they need a shared secret key. But they have never met, and everything they send — including any attempt to agree on a key — is read by an eavesdropper, Eve.

It sounds impossible: whatever Alice tells Bob, Eve hears too. In 1976 Whitfield Diffie and Martin Hellman published a way out. It needs a function that is

- **easy to compute** in one direction, and
- **hard to invert** in the other.

For ordinary numbers, exponentiation is not such a function: if Eve sees $5^x = 15{,}625$, she computes $x = \log_5 15{,}625 = 6$ instantly. The logarithm undoes the power, which has been the theme of this whole series.

But do the arithmetic **modulo a prime** $p$ — keep only remainders, like a clock — and things change. Computing

$$
5^{x} \bmod 23
$$

is still easy (fast exponentiation from notebook 04 needs about $2\log_2 x$ multiplications). Going back — given $8$, find $x$ with $5^x \equiv 8 \pmod{23}$ — has no known efficient method for well-chosen large primes. The answer exists and is called the **discrete logarithm**; it just seems to be very expensive to find. The rest of this notebook explains why, and how Alice and Bob exploit it.

**Prerequisites**

- The logarithm as the inverse of exponentiation — [The logarithm as inverse of exponentiation](../00_foundations/02_logarithm_as_inverse_of_exponentiation.ipynb).
- The laws of logarithms — [The laws of logarithms](../00_foundations/03_laws_of_logarithms.ipynb).
- Fast exponentiation by repeated squaring, and $O(\log n)$ versus $O(n)$ — [Logarithmic algorithms](04_logarithmic_algorithms.ipynb).
- Bits and key sizes: a $k$-bit number is about $2^k$ — [Bits, digits and counting](../05_information_and_probability/01_bits_digits_and_counting.ipynb).
- Modular arithmetic ("clock arithmetic") is introduced from scratch below.

After completing this notebook you will be able to:

- Compute powers modulo a prime with `pow(g, x, p)` and explain why this is fast.
- Define the discrete logarithm $\log_g h \bmod p$, find primitive roots, and check the laws of logarithms modulo $p - 1$.
- Implement brute-force search and the baby-step giant-step algorithm, and show that their costs scale like $p$ and $\sqrt p$.
- Carry out a toy Diffie–Hellman key exchange and explain what Eve would have to compute to break it.
- Explain why the real logarithm is easy (order and continuity allow bisection) while the discrete one is not.

## 2. Theory

### 2.1 Arithmetic modulo $p$

For a whole number $a$ and a modulus $p$, $a \bmod p$ is the remainder after dividing by $p$, a number in $\{0, 1, \dots, p - 1\}$. We write $a \equiv b \pmod p$ when $a$ and $b$ have the same remainder. Addition and multiplication respect remainders, so we may reduce after every step:

$$
(a \cdot b) \bmod p = \big((a \bmod p)(b \bmod p)\big) \bmod p.
$$

That is why $5^{1000} \bmod 23$ never needs the $699$-digit number $5^{1000}$: square-and-multiply, reducing after each multiplication, keeps every intermediate value below $23^2$.

### 2.2 Primitive roots

Let $p$ be prime. Fermat's little theorem says $g^{p-1} \equiv 1 \pmod p$ for every $g$ not divisible by $p$, so powers of $g$ repeat with period dividing $p - 1$. A **primitive root** (generator) is a $g$ whose powers $g^0, g^1, \dots, g^{p-2}$ run through **every** nonzero remainder exactly once. Every prime has primitive roots. For $p = 11$, $g = 2$:

| $x$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| $2^x \bmod 11$ | 1 | 2 | 4 | 8 | 5 | 10 | 9 | 7 | 3 | 6 |

All ten nonzero remainders appear, in a scrambled order.

### 2.3 The discrete logarithm

Given a primitive root $g$ modulo $p$ and a nonzero remainder $h$, the **discrete logarithm** of $h$ is the unique $x \in \{0, 1, \dots, p - 2\}$ with

$$
g^{x} \equiv h \pmod p \quad\Longleftrightarrow\quad x = \log_g h \pmod{p - 1}.
$$

It obeys the laws of logarithms, with exponents taken modulo $p - 1$ (because $g^{p-1} \equiv 1$):

$$
\log_g(ab) \equiv \log_g a + \log_g b, \qquad \log_g(a^k) \equiv k \log_g a \pmod{p - 1}.
$$

So a table of discrete logarithms would turn modular multiplication into addition — exactly Napier's idea from notebook 00/01. The catch is that for a large prime, nobody can build the table.

### 2.4 Why the real logarithm is easy and the discrete one is not

For real numbers, $b^x$ (with $b > 1$) is **increasing** and **continuous**. To solve $b^x = h$, guess $x$, compare $b^x$ with $h$, and you learn which way to go: too big means go left. That is bisection, and each step halves the interval, so every step buys one more binary digit of accuracy — the binary search of notebook 04 again. Even better methods (series, Newton's method) exploit the smoothness further.

Modulo $p$, both properties are destroyed. $g^x \bmod p$ jumps around the range with no visible pattern; $g^x \bmod p$ being **close** to $h$ says **nothing** about whether $x$ is close to the answer, and "too big" means nothing because remainders wrap around. Without order and continuity, binary search has nothing to grip. The plots in section 5 make this visible.

### 2.5 Algorithms for the discrete logarithm

**Brute force.** Try $x = 0, 1, 2, \dots$ until $g^x \equiv h$. Up to $p - 1$ multiplications: cost $O(p)$. For a $2048$-bit prime that is about $2^{2048} \approx 10^{616}$ steps.

**Baby-step giant-step** (Daniel Shanks, 1971). Let $m = \lceil \sqrt{p - 1} \rceil$ and write the unknown as $x = i\,m + j$ with $0 \le i, j < m$. Then

$$
g^{x} = h \quad\Longleftrightarrow\quad g^{j} = h \cdot \left(g^{-m}\right)^{i}.
$$

- **Baby steps:** store $g^0, g^1, \dots, g^{m-1}$ in a hash table ($m$ multiplications).
- **Giant steps:** compute $h, h\,g^{-m}, h\,g^{-2m}, \dots$ and look each one up ($\le m$ multiplications).

A match at giant step $i$ and baby step $j$ gives $x = im + j$. Total cost about $2\sqrt p$ operations and $\sqrt p$ memory — a square root, not a logarithm, of the brute-force cost. For a $2048$-bit prime that is still about $2^{1024}$.

**Better attacks.** If $p - 1$ has only small prime factors, the Pohlig–Hellman method splits the problem into small pieces (exercise 8), so real systems choose primes where $p - 1$ has a very large prime factor. For prime moduli there are also **index-calculus** methods (the number field sieve) that run in sub-exponential time — much faster than $\sqrt p$, which is why prime-field Diffie–Hellman uses $2048$-bit or larger primes. No polynomial-time classical algorithm is known; whether one exists is an open problem. (A large quantum computer running Shor's algorithm would solve discrete logarithms efficiently, which is one motivation for "post-quantum" cryptography.)

### 2.6 Diffie–Hellman key exchange

1. Alice and Bob publicly agree on a prime $p$ and a primitive root $g$.
2. Alice picks a secret $a$ and sends $A = g^{a} \bmod p$. Bob picks a secret $b$ and sends $B = g^{b} \bmod p$.
3. Alice computes $B^{a} \bmod p$; Bob computes $A^{b} \bmod p$. Both equal the **shared secret**

$$
s = g^{ab} \bmod p, \qquad\text{because}\qquad (g^{b})^{a} = g^{ab} = (g^{a})^{b}.
$$

Eve sees $p, g, A, B$. To get $s$ the obvious route is to compute $a = \log_g A \bmod p$ — a discrete logarithm. The power law $(g^b)^a = (g^a)^b$, read with exponents, is the whole trick.

Textbook Diffie–Hellman as written here is **unauthenticated**: an active attacker who can alter messages can sit in the middle and run one exchange with each side. Real protocols (TLS, SSH, Signal) combine it with signatures or other authentication, use vetted large groups or elliptic curves, and use carefully engineered libraries.

## 3. Worked Examples

### Example 1 — a discrete logarithm by table

From the table in 2.2 ($p = 11$, $g = 2$): $\log_2 9 \bmod 11 = 6$, since $2^6 = 64 = 5 \cdot 11 + 9$.

### Example 2 — the product law modulo $p$

Still $p = 11$, $g = 2$: $\log_2 3 = 8$ and $\log_2 5 = 4$. Then $3 \cdot 5 = 15 \equiv 4 \pmod{11}$ and

$$
\log_2 4 = 2 \equiv 8 + 4 = 12 \pmod{10}. \checkmark
$$

The exponents add — modulo $p - 1 = 10$.

### Example 3 — fast modular exponentiation by hand

$5^{6} \bmod 23$: $5^2 = 25 \equiv 2$, $5^4 \equiv 2^2 = 4$, $5^6 = 5^4 \cdot 5^2 \equiv 4 \cdot 2 = 8$. Three multiplications, every number tiny.

### Example 4 — a toy Diffie–Hellman exchange

Public: $p = 23$, $g = 5$. Alice's secret $a = 6$, Bob's secret $b = 15$.

- Alice sends $A = 5^{6} \bmod 23 = 8$.
- Bob sends $B = 5^{15} \bmod 23 = 19$.
- Alice computes $19^{6} \bmod 23 = 2$; Bob computes $8^{15} \bmod 23 = 2$.

Shared secret: $s = 2$. Eve saw $23, 5, 8, 19$.

### Example 5 — baby-step giant-step by hand

Find $x$ with $5^{x} \equiv 8 \pmod{23}$. Here $p - 1 = 22$, $m = \lceil\sqrt{22}\,\rceil = 5$.

- Baby steps $5^{j}$, $j = 0..4$: $1, 5, 2, 10, 4$.
- $5^{5} = 3125 \equiv 20$, and $20^{-1} \equiv 15 \pmod{23}$ (since $20 \cdot 15 = 300 = 13 \cdot 23 + 1$). So $g^{-m} \equiv 15$.
- Giant step $i = 0$: $8$ — not in the table. $i = 1$: $8 \cdot 15 = 120 \equiv 5$ — in the table at $j = 1$.

So $x = 1 \cdot 5 + 1 = 6$, Alice's secret from Example 4. Two giant steps instead of up to $22$ trials.

## 4. Python Examples

### 4.1 Modular exponentiation is fast

Python's three-argument `pow(g, x, p)` uses repeated squaring. We also write it by hand, counting multiplications, and time it on numbers with hundreds of digits.

In [ ]:
import math
import random
import time


def modular_power(base, exponent, modulus):
    # square-and-multiply, reducing after every multiplication
    result = 1
    square = base % modulus
    multiplications = 0
    while exponent > 0:
        if exponent % 2 == 1:
            result = (result * square) % modulus
            multiplications += 1
        exponent //= 2
        if exponent > 0:
            square = (square * square) % modulus
            multiplications += 1
    return result, multiplications


print("5^6  mod 23 =", modular_power(5, 6, 23))
print("5^15 mod 23 =", modular_power(5, 15, 23), "  pow:", pow(5, 15, 23))
print()

random.seed(0)
modulus = random.getrandbits(2048) | 1          # a 2048-bit odd number (not necessarily prime)
base = random.getrandbits(2000)
exponent = random.getrandbits(2048)
start = time.perf_counter()
value = pow(base, exponent, modulus)
elapsed = time.perf_counter() - start
result, count = modular_power(base, exponent, modulus)
print(f"2048-bit exponent: {count} multiplications (about 1.5 * 2048), pow took {elapsed * 1000:.2f} ms")
print("same result:", value == result, "  digits of the modulus:", len(str(modulus)))

About $3000$ multiplications of $600$-digit numbers take a few tens of milliseconds at most. The exponent has $2048$ bits, so naive repeated multiplication would need about $2^{2048}$ steps — repeated squaring replaces the exponent by its logarithm.

### 4.2 Primitive roots and a table of discrete logarithms

In [ ]:
import sympy as sp


def powers_table(generator, prime):
    return [pow(generator, x, prime) for x in range(prime - 1)]


prime = 11
for candidate in range(2, prime):
    values = powers_table(candidate, prime)
    is_generator = len(set(values)) == prime - 1
    print(f"g = {candidate:>2}: {values}  primitive root: {is_generator}")

print()
print("sympy primitive_root(11) =", sp.primitive_root(11), "  primitive_root(23) =", sp.primitive_root(23))

generator = 2
discrete_log_table = {pow(generator, x, prime): x for x in range(prime - 1)}
print("log table for g = 2 mod 11:", dict(sorted(discrete_log_table.items())))
a, b = 3, 5
print(f"log(3) + log(5) = {discrete_log_table[a] + discrete_log_table[b]}, "
      f"log(3*5 mod 11) = {discrete_log_table[(a * b) % prime]}  (equal mod {prime - 1})")

### 4.3 Brute force and baby-step giant-step

Both functions return the logarithm and the number of group operations (modular multiplications) used. We cross-check with SymPy's `discrete_log`.

In [ ]:
import math

import sympy as sp
from sympy.ntheory import discrete_log


def brute_force_log(target, generator, prime):
    value = 1
    for exponent in range(prime - 1):
        if value == target:
            return exponent, exponent      # exponent multiplications were needed to reach it
        value = (value * generator) % prime
    return None, prime - 1


def baby_step_giant_step(target, generator, prime):
    order = prime - 1
    m = math.isqrt(order)
    if m * m < order:
        m += 1
    operations = 0

    baby_steps = {}
    value = 1
    for j in range(m):
        baby_steps.setdefault(value, j)
        value = (value * generator) % prime
        operations += 1

    giant_factor = pow(generator, -m, prime)     # g^(-m) mod p (modular inverse, Python 3.8+)
    current = target % prime
    for i in range(m):
        if current in baby_steps:
            return (i * m + baby_steps[current]) % order, operations
        current = (current * giant_factor) % prime
        operations += 1
    return None, operations


print("brute force: 5^x = 8 mod 23 ->", brute_force_log(8, 5, 23))
print("BSGS       : 5^x = 8 mod 23 ->", baby_step_giant_step(8, 5, 23))
print("sympy      :", discrete_log(23, 8, 5))
print()

prime, generator = 1_000_003, 2
target = pow(generator, 765_432, prime)
print("p = 1,000,003: brute force", brute_force_log(target, generator, prime))
print("               BSGS       ", baby_step_giant_step(target, generator, prime))
print("               sympy      ", discrete_log(prime, target, generator))
print("is 2 a primitive root mod 1,000,003?", sp.is_primitive_root(2, prime))

### 4.4 How the work grows: $p$ versus $\sqrt p$

We pick primes of increasing size, a primitive root for each, and a hard target $h = g^{p-2}$ (the last exponent brute force would try). Brute force is only run up to about $10^6$; BSGS goes much further.

In [ ]:
import math
import time

import sympy as sp


def brute_force_log(target, generator, prime):
    value = 1
    for exponent in range(prime - 1):
        if value == target:
            return exponent, exponent
        value = (value * generator) % prime
    return None, prime - 1


def baby_step_giant_step(target, generator, prime):
    order = prime - 1
    m = math.isqrt(order)
    if m * m < order:
        m += 1
    operations = 0
    baby_steps = {}
    value = 1
    for j in range(m):
        baby_steps.setdefault(value, j)
        value = (value * generator) % prime
        operations += 1
    giant_factor = pow(generator, -m, prime)
    current = target % prime
    for i in range(m):
        if current in baby_steps:
            return (i * m + baby_steps[current]) % order, operations
        current = (current * giant_factor) % prime
        operations += 1
    return None, operations


print(f"{'p':>15} {'brute ops':>12} {'BSGS ops':>10} {'2*sqrt(p)':>10} {'BSGS time (s)':>14}")
for size in [10**3, 10**4, 10**5, 10**6, 10**8, 10**10, 10**11]:
    prime = sp.nextprime(size)
    generator = sp.primitive_root(prime)
    secret = prime - 2
    target = pow(generator, secret, prime)

    brute_text = "-"
    if prime < 2 * 10**6:
        found, brute_ops = brute_force_log(target, generator, prime)
        assert found == secret
        brute_text = str(brute_ops)

    start = time.perf_counter()
    found, bsgs_ops = baby_step_giant_step(target, generator, prime)
    elapsed = time.perf_counter() - start
    assert found == secret
    print(f"{prime:>15} {brute_text:>12} {bsgs_ops:>10} {2 * math.sqrt(prime):>10.0f} {elapsed:>14.3f}")

Multiplying $p$ by $100$ multiplies the brute-force work by $100$ but the BSGS work only by $10$. Still, both are **exponential in the number of digits** of $p$: adding $2$ digits multiplies the BSGS cost by $10$. Compare fast exponentiation, whose cost grows only **linearly** in the number of digits.

### 4.5 A toy Diffie–Hellman exchange — and Eve breaking it

With a prime near $10^{10}$, Eve can still run BSGS in a fraction of a second. (Secrets are drawn with Python's `random` for reproducibility; real code uses the `secrets` module and a vetted cryptographic library.)

In [ ]:
import math
import random
import time

import sympy as sp


def baby_step_giant_step(target, generator, prime):
    order = prime - 1
    m = math.isqrt(order)
    if m * m < order:
        m += 1
    baby_steps = {}
    value = 1
    for j in range(m):
        baby_steps.setdefault(value, j)
        value = (value * generator) % prime
    giant_factor = pow(generator, -m, prime)
    current = target % prime
    for i in range(m):
        if current in baby_steps:
            return (i * m + baby_steps[current]) % order
        current = (current * giant_factor) % prime
    return None


# public parameters
prime = sp.nextprime(10**10)
generator = sp.primitive_root(prime)

# private choices
rng = random.Random(42)
alice_secret = rng.randrange(2, prime - 1)
bob_secret = rng.randrange(2, prime - 1)

# public messages
alice_public = pow(generator, alice_secret, prime)
bob_public = pow(generator, bob_secret, prime)

alice_shared = pow(bob_public, alice_secret, prime)
bob_shared = pow(alice_public, bob_secret, prime)
print("public p, g     :", prime, generator)
print("Alice sends A   :", alice_public)
print("Bob sends B     :", bob_public)
print("shared secrets  :", alice_shared, bob_shared, " equal:", alice_shared == bob_shared)

start = time.perf_counter()
recovered_secret = baby_step_giant_step(alice_public, generator, prime)
elapsed = time.perf_counter() - start
eve_shared = pow(bob_public, recovered_secret, prime)
print()
print(f"Eve recovers a = {recovered_secret} in {elapsed:.2f} s (true a = {alice_secret})")
print("Eve's shared secret:", eve_shared, " matches:", eve_shared == alice_shared)

Eve's recovered exponent is congruent to Alice's modulo $p - 1$ (here they are equal), and that is all she needs. With a $2048$-bit prime, the same attack would need about $2^{1024}$ operations.

## 5. Visualizations

### 5.1 Real exponentiation is smooth; modular exponentiation is scattered

Left: $2^x$ over the reals — increasing and continuous, so it can be inverted by bisection. Right: $g^x \bmod p$ for $p = 1019$ — the same rule, but the points spray across the whole range with no order.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

prime = 1019
generator = sp.primitive_root(prime)
exponents = np.arange(prime - 1)
values = [pow(generator, int(x), prime) for x in exponents]

fig, (ax_real, ax_mod) = plt.subplots(1, 2, figsize=(13, 5))

x_real = np.linspace(0, 10, 400)
ax_real.plot(x_real, 2.0**x_real, color="tab:blue")
ax_real.axhline(600, color="crimson", linestyle="--", label="target h = 600")
ax_real.axvline(np.log2(600), color="gray", linestyle=":", label=f"x = log2(600) = {np.log2(600):.3f}")
ax_real.set_xlabel("x")
ax_real.set_ylabel("2^x")
ax_real.set_title("Real: 2^x is increasing, so log2 is easy")
ax_real.grid(True, alpha=0.3)
ax_real.legend(loc="upper left")

ax_mod.scatter(exponents, values, s=4, color="tab:orange")
ax_mod.set_xlabel("x")
ax_mod.set_ylabel(f"{generator}^x mod {prime}")
ax_mod.set_title(f"Modular: {generator}^x mod {prime} has no order to exploit")
ax_mod.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Being close tells you nothing

For the real logarithm, a guess $x$ whose power $2^x$ is close to the target is close to the answer. Modulo $p$, plot for every $x$ how far $g^x \bmod p$ is from the target $h$ against how far $x$ is from the true logarithm: a shapeless cloud, with a correlation near zero — nothing a search could use.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

prime = 1019
generator = sp.primitive_root(prime)
true_log = 400
target = pow(generator, true_log, prime)

exponents = np.arange(prime - 1)
value_distance = np.array([abs(pow(generator, int(x), prime) - target) for x in exponents])
exponent_distance = np.abs(exponents - true_log)

x_real = np.linspace(0.1, 12, 400)
real_true = np.log2(600)

fig, (ax_real, ax_mod) = plt.subplots(1, 2, figsize=(13, 5))
ax_real.plot(np.abs(x_real - real_true), np.abs(2.0**x_real - 600), ".", markersize=3, color="tab:blue")
ax_real.set_xlabel("|x - log2(600)|")
ax_real.set_ylabel("|2^x - 600|")
ax_real.set_title("Real: close in value means close in exponent")
ax_real.grid(True, alpha=0.3)

ax_mod.scatter(exponent_distance, value_distance, s=4, color="tab:orange")
ax_mod.set_xlabel(f"|x - true discrete log|   (true log = {true_log})")
ax_mod.set_ylabel(f"|{generator}^x mod {prime} - h|")
ax_mod.set_title("Modular: closeness in value carries no information")
ax_mod.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("correlation between the two distances (modular):",
      round(np.corrcoef(exponent_distance, value_distance)[0, 1], 3))

### 5.3 Cost of the attacks on log–log axes

Operation counts measured in section 4.4, with the reference lines $p$ and $2\sqrt p$. On log–log axes they are straight lines of slope $1$ and $\tfrac12$. The cost of the honest parties — one fast exponentiation, about $2\log_2 p$ multiplications — is shown for comparison: it barely rises.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp


def baby_step_giant_step_ops(target, generator, prime):
    order = prime - 1
    m = math.isqrt(order)
    if m * m < order:
        m += 1
    operations = 0
    baby_steps = {}
    value = 1
    for j in range(m):
        baby_steps.setdefault(value, j)
        value = (value * generator) % prime
        operations += 1
    giant_factor = pow(generator, -m, prime)
    current = target % prime
    for i in range(m):
        if current in baby_steps:
            return operations
        current = (current * giant_factor) % prime
        operations += 1
    return operations


primes = [sp.nextprime(10**k) for k in range(2, 11)]
bsgs_counts = []
for prime in primes:
    generator = sp.primitive_root(prime)
    bsgs_counts.append(baby_step_giant_step_ops(pow(generator, prime - 2, prime), generator, prime))

p_axis = np.logspace(2, 10, 100)
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(p_axis, p_axis, color="tab:red", label="brute force ~ p")
ax.plot(p_axis, 2 * np.sqrt(p_axis), color="tab:blue", label="baby-step giant-step ~ 2 sqrt(p)")
ax.plot(primes, bsgs_counts, "o", color="tab:blue", label="BSGS, measured")
ax.plot(primes, [p - 2 for p in primes], "s", color="tab:red", markersize=4, label="brute force, worst case p - 2")
ax.plot(p_axis, 2 * np.log2(p_axis), color="green", label="fast exponentiation ~ 2 log2(p)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("prime p (log scale)")
ax.set_ylabel("modular multiplications (log scale)")
ax.set_title("Easy direction versus hard direction")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper left")
plt.show()

### 5.4 Extrapolating to real key sizes

Using $2\sqrt{p} \approx 2^{k/2 + 1}$ operations for a $k$-bit prime and a generous $10^{9}$ operations per second. (Real attacks on prime fields use the faster number field sieve, which is why real key sizes are $2048$ bits and more rather than the $128$ bits this plot might suggest; the point here is the shape of the curves.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

bits = np.arange(16, 257, 8)
bsgs_operations_log10 = (bits / 2 + 1) * np.log10(2)
fast_power_operations_log10 = np.log10(2 * bits)
seconds_per_year = 3.15e7
age_of_universe_years = 1.4e10
bsgs_years_log10 = bsgs_operations_log10 - 9 - np.log10(seconds_per_year)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(bits, bsgs_operations_log10, color="tab:blue", label="BSGS operations (log10)")
ax.plot(bits, fast_power_operations_log10, color="green", label="fast exponentiation operations (log10)")
ax.plot(bits, bsgs_years_log10, "--", color="tab:purple", label="BSGS years at 10^9 ops/s (log10)")
ax.axhline(np.log10(age_of_universe_years), color="gray", linestyle=":", label="age of the universe in years (log10)")
ax.set_xlabel("size of the prime p in bits (linear)")
ax.set_ylabel("log10 of the count")
ax.set_title("Linear in the bits for the users, exponential in the bits for the attacker")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.show()

for k in [64, 128, 256]:
    years = 10 ** ((k / 2 + 1) * np.log10(2) - 9) / seconds_per_year
    print(f"{k:>3}-bit prime: BSGS ~ 2^{k // 2 + 1} operations ~ {years:.1e} years at 10^9 ops/s")

## 6. Exercises

### Basic

1. Compute $3^x \bmod 7$ for $x = 0, 1, \dots, 5$. Is $3$ a primitive root modulo $7$?
2. Using your table, find $\log_3 6 \bmod 7$.
3. Carry out Diffie–Hellman with $p = 23$, $g = 5$, Alice's secret $a = 4$, Bob's secret $b = 3$. What do they send, and what is the shared secret?

### Intermediate

4. **Estimate it first, then compute.** For a $64$-bit prime ($p \approx 2^{64}$), estimate the brute-force and baby-step giant-step work and the running times at $10^9$ operations per second. Then compute them. How much memory would BSGS's table need at $8$ bytes per entry?
5. Show that $2$ is **not** a primitive root modulo $7$. Which remainders have no discrete logarithm to base $2$?
6. For $p = 11$, $g = 2$, check $\log_g(ab) \equiv \log_g a + \log_g b \pmod{10}$ for all pairs $a, b$. Why is the modulus $p - 1$ and not $p$?

### Advanced

7. Prove that baby-step giant-step always finds the logarithm: show every $x \in \{0, \dots, p - 2\}$ can be written as $x = im + j$ with $0 \le i, j < m$ when $m = \lceil \sqrt{p - 1} \rceil$. Why is $m \approx \sqrt{p}$ the best choice for the total work $m + (p - 1)/m$?
8. **Pohlig–Hellman in the extreme case.** $p = 65537 = 2^{16} + 1$ is prime, so $p - 1 = 2^{16}$, and $3$ is a primitive root. Show that the discrete logarithm can be found **one bit at a time** with $16$ tests: if $x = x_0 + 2x_1 + 4x_2 + \cdots$, then $h^{(p-1)/2} \equiv (-1)^{x_0}$, which reveals $x_0$, and so on. Implement it and find $\log_3 12345 \bmod 65537$. Why must real systems avoid such primes?
9. Use bisection to compute the real logarithm $\log_2 1000$ to within $10^{-12}$, starting from the interval $[0, 20]$. How many steps does it take, and how does that number relate to $\log_2$ of (interval length / tolerance)? Explain in two sentences why the same idea cannot work modulo $p$.

## 7. Solutions

1. $3^0, \dots, 3^5 \bmod 7 = 1, 3, 2, 6, 4, 5$. All six nonzero remainders appear, so $3$ is a primitive root.
2. $3^3 = 27 = 3 \cdot 7 + 6$, so $\log_3 6 \bmod 7 = 3$.
3. $A = 5^4 \bmod 23 = 625 \bmod 23 = 4$; $B = 5^3 \bmod 23 = 125 \bmod 23 = 10$. Alice: $10^4 \bmod 23 = 18$; Bob: $4^3 \bmod 23 = 64 \bmod 23 = 18$. Shared secret $18$.
4. Brute force: about $2^{64} \approx 1.8 \times 10^{19}$ operations, $\approx 1.8 \times 10^{10}$ s, close to $600$ years. BSGS: $2\sqrt{p} = 2^{33} \approx 8.6 \times 10^{9}$ operations, about $9$ seconds — but the table has $2^{32} \approx 4.3 \times 10^9$ entries, about $34$ GB at $8$ bytes each. A $64$-bit prime is far too small.
5. $2^0, 2^1, 2^2, 2^3 \bmod 7 = 1, 2, 4, 1$: the powers cycle with period $3$ and only reach $\{1, 2, 4\}$. The remainders $3, 5, 6$ have no logarithm to base $2$.
6. The check below confirms all $100$ pairs. Exponents live modulo $p - 1$ because $g^{p-1} \equiv 1$: adding $p - 1$ to an exponent does not change the power (Fermat's little theorem).
7. Division with remainder: $x = im + j$ with $i = \lfloor x/m \rfloor$, $0 \le j < m$. Since $x \le p - 2 < m^2$, $i < m$. Baby steps cover every $j$, giant steps every $i$, so the pair is found. The total $m + (p-1)/m$ is minimised where its derivative $1 - (p-1)/m^2$ vanishes: $m = \sqrt{p - 1}$, giving about $2\sqrt{p}$.
8. Since $3^{(p-1)/2} \equiv -1$, raising $h = 3^x$ to the power $2^{15}$ gives $3^{x \cdot 2^{15}} \equiv (-1)^{x_0}$ (all higher bits contribute multiples of $2^{16}$, which vanish). Having found $x_0$, divide it out ($h \cdot 3^{-x_0}$) and raise to $2^{14}$ to find $x_1$, and so on — $16$ steps, i.e. $\log_2(p - 1)$ steps instead of $\sqrt p$. The code below finds $\log_3 12345 \bmod 65537$ and checks it. In general Pohlig–Hellman reduces the work to the size of the **largest prime factor** of $p - 1$, so real systems choose primes where $p - 1$ has a huge prime factor (for example "safe primes" $p = 2q + 1$ with $q$ prime).
9. Bisection halves the interval each step, so it needs about $\log_2(20 / 10^{-12}) \approx 44.2$, i.e. $45$ steps. It works because $2^x$ is increasing: comparing $2^{\text{mid}}$ with $1000$ tells us which half contains the answer. Modulo $p$ the comparison "$g^{\text{mid}} \bmod p < h$" says nothing about where $x$ lies, since the values wrap around and are scattered (section 5.2).

Numerical checks:

In [ ]:
import math

print("Ex 1:", [pow(3, x, 7) for x in range(6)], " primitive:", len({pow(3, x, 7) for x in range(6)}) == 6)
print("Ex 2:", [x for x in range(6) if pow(3, x, 7) == 6])
alice_public, bob_public = pow(5, 4, 23), pow(5, 3, 23)
print("Ex 3:", alice_public, bob_public, pow(bob_public, 4, 23), pow(alice_public, 3, 23))
brute_seconds = 2**64 / 1e9
print("Ex 4:", f"brute {2**64:.2e} ops = {brute_seconds / 3.15e7:.0f} years;",
      f"BSGS {2**33:.2e} ops = {2**33 / 1e9:.1f} s; table {2**32 * 8 / 1e9:.0f} GB")
print("Ex 5:", sorted({pow(2, x, 7) for x in range(6)}))

prime, generator = 11, 2
log_table = {pow(generator, x, prime): x for x in range(prime - 1)}
law_holds = all(log_table[(a * b) % prime] == (log_table[a] + log_table[b]) % (prime - 1)
                for a in range(1, prime) for b in range(1, prime))
print("Ex 6:", law_holds)

m = math.isqrt(22) + 1
print("Ex 7: p = 23, m =", m, " all x covered:",
      sorted(i * m + j for i in range(m) for j in range(m) if i * m + j <= 21) == list(range(22)))


def pohlig_hellman_power_of_two(target, generator, prime, bits):
    exponent = 0
    generator_inverse = pow(generator, -1, prime)
    for bit in range(bits):
        reduced = (target * pow(generator_inverse, exponent, prime)) % prime
        test = pow(reduced, (prime - 1) >> (bit + 1), prime)
        if test != 1:          # test is -1 (i.e. prime - 1): this bit is set
            exponent += 1 << bit
    return exponent


x = pohlig_hellman_power_of_two(12345, 3, 65537, 16)
print("Ex 8: log_3 12345 mod 65537 =", x, " check:", pow(3, x, 65537) == 12345)

low, high, steps = 0.0, 20.0, 0
while high - low > 1e-12:
    middle = (low + high) / 2
    if 2**middle < 1000:
        low = middle
    else:
        high = middle
    steps += 1
print("Ex 9:", steps, "steps;", round(low, 12), "vs", round(math.log2(1000), 12),
      "; log2(20 / 1e-12) =", round(math.log2(20 / 1e-12), 2))

## 8. Conclusion

- Modulo a prime $p$, a primitive root $g$ makes $x \mapsto g^x \bmod p$ a one-to-one map of $\{0, \dots, p - 2\}$ onto the nonzero remainders. Its inverse is the **discrete logarithm**: $g^x \equiv h \iff x = \log_g h \pmod{p - 1}$, and it obeys the usual laws with exponents modulo $p - 1$.
- The forward direction is **easy**: repeated squaring needs about $2\log_2 x$ multiplications, so `pow(g, x, p)` handles 2048-bit numbers in milliseconds.
- The backward direction seems **hard**: brute force costs about $p$, baby-step giant-step about $2\sqrt p$ — both exponential in the number of digits. Better methods exist for prime fields (and for special primes, Pohlig–Hellman), but no efficient classical algorithm is known.
- The real logarithm is easy because $b^x$ is **ordered and continuous**, so bisection and series work. $g^x \bmod p$ is neither: its values scatter, and nearness in value says nothing about nearness in exponent.
- **Diffie–Hellman**: Alice sends $g^a$, Bob sends $g^b$, both compute $g^{ab}$; Eve would need a discrete logarithm. Real deployments add authentication and use large, vetted parameters or elliptic curves — the toy sizes here are for learning only.

**Next:** module 07 leaves the real numbers in a different direction — notebook 07/01 asks what $\ln(-1)$ could mean, and finds the answer $i\pi$ in the complex plane: [Logarithms of complex numbers](../07_beyond_the_reals/01_logarithms_of_complex_numbers.ipynb).